In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import array_contains, col, explode

spark = (SparkSession.builder
         .appName("filter-data")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

In [0]:
df = (spark.read.format("csv")
      .option("header", "true")
      .option("nullValue", "null")
      .option("dateFormat", "LLLL d, y")
      .load("../data/netflix_titles.csv"))

In [0]:
filtered_df = df.filter(col("release_year") > 2020)
filtered_df.show()

In [0]:
filtered_df = (
    df.filter(
        (col("country") == "United States")
        & (col("release_year") > 2020)))

filtered_df.show()

In [0]:
filtered_df = (
    df.filter(
        col("country")
        .isin(["United States", "United Kingdom",  "India"])))
filtered_df.show(3)

### Filtering on string

In [0]:
# filter the DataFrame based on a substring match
filtered_df = df.filter(col("listed_in").like("%Crime%"))

# display the filtered DataFrame
filtered_df.show()

In [0]:
# filter the DataFrame based on a regular expression match
filtered_df = df.filter(col("listed_in").rlike("(Crime|Thrillers)"))

# display the filtered DataFrame
filtered_df.show()

### Filtering on Data Ranges

In [0]:
# filter the DataFrame based on a date range
filtered_df = df.filter((col("date_added") >= "2021-09-05") & (col("date_added") <= "2021-09-01"))

# display the filtered DataFrame
filtered_df.show()

In [0]:
# filter the DataFrame based on a date range
filtered_df = df.filter((col("date_added").between("2021-02-01","2021-03-01")))

# display the filtered DataFrame
filtered_df.show()

### Filter on Arrays

In [0]:
# Read parquet file into a DataFrame
df_recipes = (spark.read
      .format("parquet")
      .load("../data/recipes.parquet"))

# filter the DataFrame based on a value in the array column
filtered_df = df_recipes.filter(array_contains(col("RecipeIngredientParts"), "apple"))

# display the filtered DataFrame
filtered_df.show()


### Filtering on map columns

In [0]:
# Read JSON file into a DataFrame
df_nobel_prizes = (spark.read
      .format("json")
      .option("multiLine", "true")
      .load("../data/nobel_prizes.json"))

df_nobel_prizes_exploded = (
    df_nobel_prizes
    .withColumn("laureates",explode(col("laureates"))) # Explode the laureates array column into rows
    .select(col("category")
            , col("year")
            , col("overallMotivation")
            , col("laureates"))) # Use dot notion for columns in the STRUCT field

filtered_df = (
    df_nobel_prizes_exploded
    .filter(
        (col("laureates").getItem("firstname") == "Albert") 
        & (col("laureates").getItem("surname") == "Einstein")))

filtered_df.show(truncate=False)

In [0]:
spark.stop()